# Notebook app examples with gradio and dash

This notebook defines a lightweight Gradio predictor and a Dash explorer for the sklearn wine dataset.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
import gradio as gr
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

wine = load_wine(as_frame=True)
df = wine.frame.copy()
feature_names = list(wine.feature_names)
df['target_name'] = df['target'].map(dict(enumerate(wine.target_names)))

X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.25, random_state=42, stratify=wine.target
)
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
selected_features = ['alcohol', 'malic_acid', 'ash', 'color_intensity']
feature_ranges = {
    feature: (float(df[feature].min()), float(df[feature].max()))
    for feature in selected_features
}

def predict_wine(alcohol, malic_acid, ash, color_intensity):
    sample = df[feature_names].median().to_frame().T
    sample.loc[:, selected_features] = [alcohol, malic_acid, ash, color_intensity]
    prediction = model.predict(sample)[0]
    return wine.target_names[prediction]

gradio_demo = gr.Interface(
    fn=predict_wine,
    inputs=[
        gr.Slider(*feature_ranges['alcohol'], label='alcohol'),
        gr.Slider(*feature_ranges['malic_acid'], label='malic_acid'),
        gr.Slider(*feature_ranges['ash'], label='ash'),
        gr.Slider(*feature_ranges['color_intensity'], label='color_intensity'),
    ],
    outputs=gr.Label(label='Predicted cultivar'),
    title='Wine cultivar predictor',
    description='Adjust a few features to predict the wine cultivar.'
)
gradio_demo

In [ ]:
dash_app = Dash(__name__)
dash_app.layout = html.Div([
    html.H2('Wine feature explorer'),
    dcc.Dropdown(feature_names, 'alcohol', id='x-axis'),
    dcc.Dropdown(feature_names, 'color_intensity', id='y-axis'),
    dcc.Graph(id='wine-scatter'),
])

@dash_app.callback(
    Output('wine-scatter', 'figure'),
    Input('x-axis', 'value'),
    Input('y-axis', 'value'),
)
def update_scatter(x_axis, y_axis):
    return px.scatter(df, x=x_axis, y=y_axis, color='target_name', hover_name='target_name')

dash_app

In [ ]:
# Optional notebook launch cells:
# gradio_demo.launch()
# dash_app.run(jupyter_mode='inline')